In [ ]:
import pyspark.sql.functions as f
from pyspark.sql import Window

Let's keep practicing with the FruitShop dataset!

Load the data and recall the schema.

In [ ]:
df_fruitshop = spark.read.parquet('/Volumes/workspace/default/class/fruitshop.parquet')
df_fruitshop.printSchema()
df_fruitshop.display()

Answer the following questions. Good luck!

7. For each order, what is the total cost? (sum of price * quantity for all items)

In [ ]:
(
    df_fruitshop
    .select('order_id', f.inline('items'))
    .withColumn('line_cost', f.col('price') * f.col('quantity'))
    .groupBy('order_id')
    .agg(f.sum('line_cost').alias('total_cost'))
).display()

8. For each order, how many items have a quantity greater than 1?

In [ ]:
(
    df_fruitshop
    .select('order_id', f.inline('items'))
    .filter(f.col('quantity') > 1)
    .groupBy('order_id')
    .agg(f.count('name').alias('items_qty_gt_1'))
).display()

9. What is the most expensive item across all orders? Return its name and price.

In [ ]:
(
    df_fruitshop
    .select(f.inline('items'))
    .orderBy(f.desc('price'))
    .limit(1)
    .select('name', 'price')
).display()

10. For each order, create a column `item_names_upper` with the item names in uppercase.

***Hint:*** Use transform with f.upper().

In [ ]:
(
    df_fruitshop
    .withColumn(
        'item_names_upper',
        f.transform(f.col('items'), lambda x: f.upper(x['name']))
    )
    .select('order_id', 'item_names_upper')
).display()

11. Which orders have more discounted items than non-discounted items?

***Hint:*** Compare size of items_discount with size of items.

In [ ]:
(
    df_fruitshop
    .filter(
        f.size(f.col('items_discount')) > (f.size(f.col('items')) - f.size(f.col('items_discount')))
    )
    .select('order_id', 'items', 'items_discount')
).display()

12. For each order, what is the average price of the discounted items only?

***Hint:*** Explode items, filter by array_contains on items_discount, then avg.

In [ ]:
(
    df_fruitshop
    .select(
        'order_id',
        'items_discount',
        f.explode('items').alias('item')
    )
    .filter(f.array_contains(f.col('items_discount'), f.col('item.name')))
    .groupBy('order_id')
    .agg(f.avg('item.price').alias('avg_discounted_price'))
).display()

13. **BONUS** For each order, rank the items by price descending using a window function and return only the top 2 most expensive items per order.

In [ ]:
window = Window.partitionBy('order_id').orderBy(f.desc('price'))

(
    df_fruitshop
    .select('order_id', f.inline('items'))
    .withColumn('rank', f.dense_rank().over(window))
    .filter(f.col('rank') <= 2)
    .select('order_id', 'name', 'price', 'rank')
).display()